# Task 152: AI-Based Shelf Inventory Monitoring System

This notebook demonstrates the implementation and execution of the **AI-Based Shelf Inventory Monitoring System** (Task 152). 

### Objective:
Detect empty or low-stock shelves using computer vision, track items and customer interactions, and maintain a robust audit log with evidence snapshots.

### Core Requirements:
- Detect relevant objects/people (YOLOv8)
- Track objects across frames (Class-Aware IoU Tracker)
- Analyze task-specific behavior (Empty/Low-stock thresholds, Stock depletion/replenishment)
- Generate alerts and store evidence (JPG snapshots at state transition points)
- Maintain event records (incident_log.csv and summary_stats.json)
- Output annotated video with live status overlays

---

## 1. Environment Setup & Dependency Installation

First, we install the required packages: `ultralytics`, `pandas`, `matplotlib`, and `opencv-python`.

In [ ]:
!pip install -q ultralytics pandas matplotlib opencv-python

## 2. Imports and Helper Verification

In [ ]:
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Set paths relative to execution
code_dir = os.getcwd()
project_dir = os.path.dirname(code_dir)
inputs_dir = os.path.join(project_dir, "Inputs")
outputs_dir = os.path.join(project_dir, "Outputs")
evidence_dir = os.path.join(outputs_dir, "evidence_frames")
inspect_dir = os.path.join(outputs_dir, "inspect_frames")

print(f"Project directory resolved to: {project_dir}")

## 3. Run Pipeline on CCTV Test Videos

We run `detect.py` CLI script on the 3 real-world test videos. Each run processes the video, performs YOLOv8 inference, updates trackers, logs events, saves annotated videos, and generates the dashboard.

In [ ]:
# Run on Video 1: Supermarket shelf monitoring (Stationary camera, customer purchase & interaction)
!python detect.py --video ../Inputs/shelf_0.mp4

# Run on Video 2: Retail store shelf with low-stock conditions
!python detect.py --video ../Inputs/shelf_1.mp4

# Run on Video 3: Restocking/stock replenishment scenario
!python detect.py --video ../Inputs/shelf_2.mp4

## 4. Verify Summary Statistics

Let's load the generated `summary_stats.json` to verify the overall results of the monitoring run.

In [ ]:
summary_path = os.path.join(outputs_dir, "summary_stats.json")
if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        stats = json.load(f)
    print("=== Inventory Summary Stats ===")
    print(json.dumps(stats, indent=4))
else:
    print("Error: summary_stats.json not found!")

## 5. View Incident Log Database

Here, we inspect the records inside `incident_log.csv` which tracks every shelf stock transition and customer interaction event with exact frame numbers and details.

In [ ]:
csv_path = os.path.join(outputs_dir, "incident_log.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Total logged events: {len(df)}")
    # Display the first 15 events
    display(df.head(15))
else:
    print("Error: incident_log.csv not found!")

## 6. Display Analytics Dashboard

We visualize the premium-designed analytics dashboard showing the distribution of events and stock timeline trends across shelves.

In [ ]:
dashboard_path = os.path.join(outputs_dir, "analytics_dashboard.png")
if os.path.exists(dashboard_path):
    display(Image(filename=dashboard_path))
else:
    print("Error: analytics_dashboard.png not found!")

## 7. Inspect Sample Frame Overlays & Evidence Snapshots

Let's display some sample frame outputs showing the active shelf overlays, item detections, customer purple boxes, and alerts.

In [ ]:
# List first few inspect frames
frames = [f for f in os.listdir(inspect_dir) if f.endswith('.jpg')]
frames.sort(key=lambda x: int(os.path.splitext(x)[0].split('_')[1]))

print(f"Found {len(frames)} inspect frames. Displaying select samples:")
for f in frames[:3]:
    print(f"\n--- Frame: {f} ---")
    display(Image(filename=os.path.join(inspect_dir, f), width=640))

### Sample Evidence Snapshot

Here we show a sample evidence frame saved during a critical stock event (Empty or Low Stock).

In [ ]:
evidence_files = [f for f in os.listdir(evidence_dir) if f.endswith('.jpg')]
if evidence_files:
    sample_ev = evidence_files[0]
    print(f"Displaying evidence frame: {sample_ev}")
    display(Image(filename=os.path.join(evidence_dir, sample_ev), width=640))
else:
    print("No evidence snapshots saved.")